# Tugas Akhir 07 (TA-07): Implementasi Q-Learning untuk Skenario RL Diskrit
**Nama:** Diaz Dynasti Saputra  
**NIM:** 301240026  
**Kelas:** 5B  
**Skenario:** Smart Courier Grid Routing (Grid 4x4, 32 States, 6 Actions)

## Langkah 1: Persiapan Lingkungan Simulasi (`SmartCourierEnv`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class SmartCourierEnv:
    def __init__(self):
        self.grid_size = 4
        self.pickup_pos = (0, 3)
        self.dropoff_pos = (3, 3)
        self.obstacles = {(1, 1), (2, 2)}
        self.n_states = 32  # 16 koordinat grid x 2 status paket
        self.n_actions = 6  # 0:Atas, 1:Bawah, 2:Kiri, 3:Kanan, 4:Pickup, 5:Dropoff
        self.reset()

    def reset(self):
        self.agent_pos = (0, 0)
        self.has_package = 0
        return self._get_state()

    def _get_state(self):
        pos_idx = self.agent_pos[0] * self.grid_size + self.agent_pos[1]
        return pos_idx * 2 + self.has_package

    def step(self, action):
        r, c = self.agent_pos
        reward = -1  # Penalti setiap langkah bergerak
        done = False

        if action in [0, 1, 2, 3]:
            nr, nc = r, c
            if action == 0: nr -= 1      # Atas
            elif action == 1: nr += 1    # Bawah
            elif action == 2: nc -= 1    # Kiri
            elif action == 3: nc += 1    # Kanan

            if 0 <= nr < self.grid_size and 0 <= nc < self.grid_size:
                if (nr, nc) in self.obstacles:
                    reward = -10  # Menabrak zona macet/hambatan
                else:
                    self.agent_pos = (nr, nc)
            else:
                reward = -5  # Menabrak batas kota

        elif action == 4:  # Pickup
            if self.agent_pos == self.pickup_pos and self.has_package == 0:
                self.has_package = 1
                reward = 20
            else:
                reward = -10

        elif action == 5:  # Dropoff
            if self.agent_pos == self.dropoff_pos and self.has_package == 1:
                reward = 50
                done = True
            else:
                reward = -10

        return self._get_state(), reward, done

env = SmartCourierEnv()
print(f"State Space: {env.n_states}, Action Space: {env.n_actions}")

## Langkah 2: Implementasi Pelatihan Q-Learning (`Epsilon-Greedy`)

In [ ]:
def train_q_learning(alpha=0.8, gamma=0.9, epsilon_start=1.0, min_epsilon=0.01, decay_rate=0.003, episodes=2000, max_steps=100, seed=301240026):
    np.random.seed(seed)
    env = SmartCourierEnv()
    q_table = np.zeros((env.n_states, env.n_actions))
    epsilon = epsilon_start
    rewards_per_episode = []

    for ep in range(episodes):
        state = env.reset()
        total_reward = 0
        for _ in range(max_steps):
            if np.random.uniform(0, 1) < epsilon:
                action = np.random.randint(0, env.n_actions)
            else:
                action = np.argmax(q_table[state, :])

            next_state, reward, done = env.step(action)
            best_next_q = np.max(q_table[next_state, :])
            q_table[state, action] = (1 - alpha) * q_table[state, action] + alpha * (reward + gamma * best_next_q)

            state = next_state
            total_reward += reward
            if done:
                break

        rewards_per_episode.append(total_reward)
        epsilon = min_epsilon + (epsilon_start - min_epsilon) * np.exp(-decay_rate * ep)

    return q_table, rewards_per_episode

q_main, rewards_main = train_q_learning(alpha=0.8, gamma=0.9, episodes=2000)
print("Q-Table akhir (5 state pertama):")
print(np.round(q_main[:5], 2))

## Langkah 3: Visualisasi Kurva Pembelajaran (Moving Average 50 Episode)

In [ ]:
plt.figure(figsize=(10, 5))
ma_main = pd.Series(rewards_main).rolling(window=50, min_periods=1).mean()
plt.plot(ma_main, color='#1E40AF', linewidth=2, label='Moving Average Reward (Window=50)')
plt.axhline(y=64, color='#059669', linestyle='--', linewidth=1.5, label='Reward Optimal Teoretis (64)')
plt.title('Kurva Pembelajaran Q-Learning - Smart Courier Grid Routing', fontsize=12, fontweight='bold')
plt.xlabel('Episode')
plt.ylabel('Rata-Rata Total Reward')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Rata-rata reward 100 episode terakhir: {np.mean(rewards_main[-100:]):.2f}")

## Langkah 4: Ekstraksi & Visualisasi Kebijakan Optimal (Policy)

In [ ]:
# Evaluasi langkah optimal
env_eval = SmartCourierEnv()
s = env_eval.reset()
action_names = {0: 'Atas', 1: 'Bawah', 2: 'Kiri', 3: 'Kanan', 4: 'Pickup', 5: 'Dropoff'}
actions_taken = []
total_eval_reward = 0

for _ in range(30):
    a = int(np.argmax(q_main[s, :]))
    actions_taken.append(action_names[a])
    s, r, done = env_eval.step(a)
    total_eval_reward += r
    if done:
        break

print("Urutan aksi optimal:", actions_taken)
print(f"Total langkah: {len(actions_taken)} | Total Reward: {total_eval_reward}")

# Visualisasi Grid Kebijakan
arrow_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→', 4: 'P', 5: 'D'}
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for pkg_status, title in [(0, 'Fase 1: Menuju Pickup (Paket = 0)'), (1, 'Fase 2: Menuju Dropoff (Paket = 1)')]:
    ax = axes[pkg_status]
    grid_vals = np.zeros((4, 4))
    for r in range(4):
        for c in range(4):
            state_idx = (r * 4 + c) * 2 + pkg_status
            best_act = int(np.argmax(q_main[state_idx, :]))
            grid_vals[r, c] = best_act
            if (r, c) in {(1, 1), (2, 2)}:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='#1E293B'))
                ax.text(c, r, 'X\n(Macet)', ha='center', va='center', color='white', fontweight='bold')
            else:
                lbl = arrow_symbols[best_act]
                if (r, c) == (0, 3): lbl += '\n[Pickup]'
                elif (r, c) == (3, 3): lbl += '\n[Dropoff]'
                elif (r, c) == (0, 0) and pkg_status == 0: lbl += '\n[Start]'
                ax.text(c, r, lbl, ha='center', va='center', color='#0F172A', fontsize=11, fontweight='bold')

    ax.imshow(grid_vals, cmap='Blues', alpha=0.35, vmin=0, vmax=5)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
    ax.grid(which='minor', color='#64748B', linestyle='-', linewidth=1.2)
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.show()

## Langkah 5: Eksperimen Hiperparameter (`Alpha = 0.8` vs `Alpha = 0.005`)

In [ ]:
_, r_fast_a = train_q_learning(alpha=0.8, gamma=0.9, decay_rate=0.01, episodes=1000)
_, r_slow_a = train_q_learning(alpha=0.005, gamma=0.9, decay_rate=0.01, episodes=1000)

plt.figure(figsize=(10, 5))
ma_fa = pd.Series(r_fast_a).rolling(window=50, min_periods=1).mean()
ma_sa = pd.Series(r_slow_a).rolling(window=50, min_periods=1).mean()

plt.plot(ma_fa, color='#1E40AF', linewidth=2, label='Alpha = 0.8 (Learning Rate Tinggi - Cepat Konvergen)')
plt.plot(ma_sa, color='#DC2626', linewidth=2, label='Alpha = 0.005 (Learning Rate Rendah - Lambat Konvergen)')
plt.title('Perbandingan Kurva Pembelajaran: Alpha = 0.8 vs Alpha = 0.005', fontsize=12, fontweight='bold')
plt.xlabel('Episode')
plt.ylabel('Rata-Rata Total Reward (MA-50)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()